In [3]:

import os
import json
import tensorflow as tf # pyright: ignore[reportMissingImports]
from tensorflow.keras import layers, Model # pyright: ignore[reportMissingImports]
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau # pyright: ignore[reportMissingImports]


DATA_DIR = os.path.join("/home/apurba-roy/Developments/FinalYear-Project/Breastcancer model/backend/classification_model", "new_data")
BATCH_SIZE = 16
IMG_SIZE = (224, 224)
EPOCHS = 50
NUM_CLASSES = 3
OUT_MODEL = "model_v1.keras"
OUT_MODEL_FALLBACK = "model_best.h5"
CLASS_JSON = "class_indices.json"
LEARNING_RATE = 1e-7


def get_datasets():
    train_ds = tf.keras.preprocessing.image_dataset_from_directory( # type: ignore
        os.path.join(DATA_DIR, "train"),
        label_mode="categorical",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=123,
        color_mode="rgb"
    )
    val_ds = tf.keras.preprocessing.image_dataset_from_directory( # type: ignore
        os.path.join(DATA_DIR, "val"),
        label_mode="categorical",
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
        color_mode="rgb"
    )

    # augmentation pipeline (applied only to train dataset)
    data_augmentation = tf.keras.Sequential([  
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.06),   # ~ +/- 6%
        tf.keras.layers.RandomZoom(0.08),
        tf.keras.layers.RandomContrast(0.08),
    ], name="data_augmentation")

    # keep class names before wrapping/prefetching
    class_names = getattr(train_ds, "class_names", None)

    AUTOTUNE = tf.data.AUTOTUNE

    # apply augmentation to training set only
    def augment(images, labels):
        # images are uint8 in [0,255] from image_dataset_from_directory
        images = tf.cast(images, tf.float32)
        images = data_augmentation(images, training=True)
        return images, labels

    train_ds = train_ds.map(augment, num_parallel_calls=AUTOTUNE)
    train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
    return train_ds, val_ds, class_names

def build_model():
    base = tf.keras.applications.EfficientNetB0(include_top=False, input_shape=IMG_SIZE + (3,), weights='imagenet') # type: ignore
    base.trainable = False
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(3, activation='softmax')(x)
    model = Model(inputs=base.input, outputs=outputs)
    return model

def main():
    train_ds, val_ds, class_names = get_datasets()
   
    if class_names is None:
        
        class_names = getattr(train_ds, "class_names", None)
    if class_names is None:
        raise RuntimeError(f"Could not determine class names. Ensure '{os.path.join(DATA_DIR,'train')}' contains one subfolder per class with images.")
    class_indices = {name: i for i, name in enumerate(class_names)}
    with open(CLASS_JSON, "w") as f:
        json.dump(class_indices, f)
    print("Class mapping saved to", CLASS_JSON, class_indices)

    model = build_model()
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), # type: ignore
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    

    callbacks = [
        ModelCheckpoint(OUT_MODEL, monitor="val_accuracy", save_best_only=True, verbose=1),
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1)
    ]

    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks)
    print("Training finished. Best model saved at:", OUT_MODEL)


if __name__ == "__main__":
    main()


Found 624 files belonging to 3 classes.
Found 155 files belonging to 3 classes.
Class mapping saved to class_indices.json {'benign': 0, 'malignant': 1, 'normal': 2}
Epoch 1/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step - accuracy: 0.5063 - loss: 1.0449
Epoch 1: val_accuracy improved from None to 0.41935, saving model to model_v1.keras

Epoch 1: finished saving model to model_v1.keras
39/39 ━━━━━━━━━━━━━━━━━━━━ 12s 202ms/step - accuracy: 0.5064 - loss: 1.0398 - val_accuracy: 0.4194 - val_loss: 1.0504 - learning_rate: 1.0000e-07
Epoch 2/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.5013 - loss: 1.0433
Epoch 2: val_accuracy did not improve from 0.41935
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 144ms/step - accuracy: 0.5096 - loss: 1.0400 - val_accuracy: 0.4194 - val_loss: 1.0500 - learning_rate: 1.0000e-07
Epoch 3/50
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step - accuracy: 0.5085 - loss: 1.0400
Epoch 3: val_accuracy did not improve from 0.41935
39/39 ━━━━━━━━━━━━━━━━━━━━ 6s 144ms/step - accurac